# 03. 데이터 전처리

01에서 수집한 JSON에서 5개 섹션(II, IV, VI, VIII, XI)을 합쳐 기업-연도 1행짜리 문서를 만들고, 다음 세 단계 정제를 거칩니다.

1. `raw_text` — 5개 섹션을 그대로 이어붙인 원문
2. `clean_text` — 한글/영문만 남기고 **로마숫자 섹션 번호(II, IV, VI, VIII, XI, ii, i, xi 등) 제거**, 회사명·일반어 정리
3. `kiwi_text` — Kiwi 형태소 분석 → 명사/용언/영문(SL)만 남기고 **불용어 제거** → 공백 구분 토큰 문자열

**버그 수정 내용** (기존 노트북 대비)
- 기존 `clean_korean_english_only`는 `[가-힣A-Za-z]+` 패턴으로 영문 토큰을 그대로 보존해 `II`, `IV`, `XI` 같은 섹션 번호 로마숫자가 살아남았음
- 또한 공백 없이 모두 이어붙여 토큰 경계가 무너졌음
- 본 노트북은 (a) 로마숫자 표기를 raw 단계에서 제거하고, (b) `clean_text`는 공백을 보존하며, (c) Kiwi 토큰화 후 길이 ≤ 1·불용어·짧은 영문 약자 추가 필터를 적용

**산출 CSV** (1개): `data/processed/dart_preprocessed.csv` — `회사이름`, `esg_year`, `raw_text`, `clean_text`, `kiwi_text`


## 1. 환경 설정과 입력 파일 확인


In [1]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
from kiwipiepy import Kiwi

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = DATA_PROCESSED / 'dart_reports'
STOPWORDS_PATH = DATA_RAW / 'stopwords_ko_esg.txt'
OUTPUT_CSV = DATA_PROCESSED / 'dart_preprocessed.csv'

SECTION_LABELS = ['II', 'IV', 'VI', 'VIII', 'XI']

print('JSON 폴더:', REPORT_DIR, '— 파일 수:', len(list(REPORT_DIR.glob('*.json'))))
print('불용어:', STOPWORDS_PATH, '— exists:', STOPWORDS_PATH.exists())
print('출력 CSV:', OUTPUT_CSV)


JSON 폴더: /Users/home1/esg-dart-project/data/processed/dart_reports — 파일 수: 381
불용어: /Users/home1/esg-dart-project/data/raw/stopwords_ko_esg.txt — exists: True
출력 CSV: /Users/home1/esg-dart-project/data/processed/dart_preprocessed.csv


## 2. JSON 로딩과 5개 섹션 결합 → `raw_text`

JSON 한 개당 1행씩 데이터프레임으로 만들고, II → IV → VI → VIII → XI 순서로 빈 줄 2개로 구분해 결합합니다. `status='OK'`가 아닌 행은 `raw_text=''`로 남깁니다.


In [2]:
def load_report(path: Path) -> dict:
    payload = json.loads(path.read_text(encoding='utf-8'))
    sections = payload.get('sections') or {}
    row = {
        '회사이름': payload.get('company_name'),
        'stock_code': str(payload.get('stock_code', '')).zfill(6),
        'fiscal_year': payload.get('fiscal_year'),
        'esg_year': payload.get('esg_year'),
        'status': payload.get('status'),
    }
    section_texts = []
    for label in SECTION_LABELS:
        text = sections.get(label) or ''
        if text.strip():
            section_texts.append(text.strip())
    row['raw_text'] = '\n\n'.join(section_texts)
    return row


report_files = sorted(REPORT_DIR.glob('*.json'))
df = pd.DataFrame(load_report(p) for p in report_files)
df['raw_text_len'] = df['raw_text'].str.len()

print(f'전체 행: {len(df)}')
print(f'raw_text 비어있는 행: {(df["raw_text_len"] == 0).sum()}')
print(f'raw_text 평균 길이: {df["raw_text_len"].mean():.0f}')
df[['회사이름', 'esg_year', 'status', 'raw_text_len']].head()


전체 행: 381
raw_text 비어있는 행: 0
raw_text 평균 길이: 102483


,회사이름,esg_year,status,raw_text_len
0,동화약품,2023,OK,47541
1,동화약품,2024,OK,48664
2,동화약품,2025,OK,50171
3,KR모터스,2023,OK,26506
4,KR모터스,2024,OK,30540


## 3. 로마숫자 섹션 번호 제거

DART 사업보고서에는 섹션 헤더(`II. 사업의 내용`)와 하위 섹션 번호(`(가)`, `i.`, `ii.`, `xi.`)가 본문에 섞여 있습니다. Kiwi 토큰화 시 `I`, `II`, `IV`, `VI`, `VIII`, `XI`, `ii`, `xi` 등이 영문 토큰(SL)으로 잡혀 TF-IDF 결과에 잡음을 만듭니다.

**전략**: `clean_text` 단계에서 로마숫자 표기를 정규식으로 제거합니다. 일반 영문(예: `ESG`, `CO2`)은 보존합니다.

- 유니코드 로마숫자(`Ⅰ Ⅱ ... Ⅻ`, `ⅰ ⅱ ... ⅻ`)
- 라틴 로마숫자 + 마침표/괄호(`I.`, `II.`, `(IV)`, `xi)`)  
- 단독 로마숫자(공백·구두점에 둘러싸인 `II`, `iv`)


In [3]:
ROMAN_UNICODE_CHARS = 'ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩⅪⅫⅰⅱⅲⅳⅴⅵⅶⅷⅸⅹⅺⅻ'
ROMAN_TOKEN = r'(?:VIII|VII|XII|XI|IV|IX|III|II|VI|V|X|I|viii|vii|xii|xi|iv|ix|iii|ii|vi|v|x|i)'

# 1) 로마숫자 + 마침표/괄호 (예: II., (IV), xi))
ROMAN_NUMBERED_RE = re.compile(rf'(?<![A-Za-z]){ROMAN_TOKEN}\s*[\.\)]')
# 2) 공백/문장경계로 둘러싸인 단독 로마숫자
ROMAN_STANDALONE_RE = re.compile(rf'(?<![A-Za-z]){ROMAN_TOKEN}(?![A-Za-z])')
# 3) 한글 섹션 번호: 가., 나., (가), (1), ①
KO_BULLET_RE = re.compile(r'\([가-힣]\)|\([0-9]+\)|[①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮]|[가-힣]\.\s')
# 4) 유니코드 로마숫자 단일 문자
ROMAN_UNICODE_RE = re.compile(rf'[{ROMAN_UNICODE_CHARS}]')


def strip_section_markers(text: str) -> str:
    if not isinstance(text, str) or not text:
        return ''
    text = ROMAN_UNICODE_RE.sub(' ', text)
    text = ROMAN_NUMBERED_RE.sub(' ', text)
    text = ROMAN_STANDALONE_RE.sub(' ', text)
    text = KO_BULLET_RE.sub(' ', text)
    return text


# 빠른 sanity check
sample = 'II. 사업의 내용\n(가) 환경경영 ① 탄소중립 i. 재생에너지 xi) 폐기물 ESG와 CO2 감축'
print('원문:', sample)
print('제거 후:', strip_section_markers(sample))


원문: II. 사업의 내용
(가) 환경경영 ① 탄소중립 i. 재생에너지 xi) 폐기물 ESG와 CO2 감축
제거 후:   사업의 내용
  환경경영   탄소중립   재생에너지   폐기물 ESG와 CO2 감축


## 4. `clean_text` 생성 — 기본 전처리

1. NFKC 정규화 (전각·호환 문자 정리)
2. 로마숫자·한글 섹션 번호 제거 (위 3절)
3. 한글·영문·숫자·공백 외 모두 제거 (특수문자·표 기호 제거)
4. 공백 정규화 (`\s+` → 단일 공백)
5. 소문자화

**기존 버그 대비**: 공백을 살려 토큰 경계가 무너지지 않도록 합니다.


In [4]:
KEEP_CHARS_RE = re.compile(r'[^가-힣A-Za-z0-9\s]+')
WHITESPACE_RE = re.compile(r'\s+')


def to_clean_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ''
    text = unicodedata.normalize('NFKC', text)
    text = strip_section_markers(text)
    text = KEEP_CHARS_RE.sub(' ', text)
    text = WHITESPACE_RE.sub(' ', text).strip().lower()
    return text


df['clean_text'] = df['raw_text'].map(to_clean_text)
df['clean_text_len'] = df['clean_text'].str.len()

print(f'clean_text 비어있는 행: {(df["clean_text_len"] == 0).sum()}')
print(f'clean_text 평균 길이: {df["clean_text_len"].mean():.0f}')

# 로마숫자 제거가 잘 됐는지 확인
sample_clean = df.loc[df['clean_text_len'] > 0, 'clean_text'].iloc[0][:500]
print('\n샘플 clean_text 앞 500자:')
print(sample_clean)


clean_text 비어있는 행: 0
clean_text 평균 길이: 88267

샘플 clean_text 앞 500자:
사업의 내용 1 사업의 개요 1 일반적인 사항지배기업인 연결실체는 제공하는 재화나 용역에 근거하여 영업부문을 구분하고 각 부문의 재무정보를 내부관리 목적으로 활용하고 있는 바 당사는 제약업 의료기기 제조 및 판매업 기타금융업을 운영하고 있습니 영위하는 영업부문 중 기타금융업의 경우 중요성의 관점에서 비중이 크지 않습니 2 지배회사의 현황 영업개황 및 사업부문의 구분 연결기준 당사의 연결실체는 94기 매출액은 340 426백만원 연결기준 으로 전년 동기 대비 16 2 증가하였으며 영업이익은 29 915백만원 연결기준 으로 전년 동기 대비 33 0 증가하였습니 제공하는 재화나 용역에 근거하여 영업부문을 구분하고 각 부문의 재무정보를 내부관리 목적으로 활용하고 있는 바 연결실체는 제약업 기타금융업 의료기기 제조판매업 등을 영위하고 있습니 연결기업은 제공하는 재화나 용역에 근거하여 영업부문을 구분하고 각 부문의 재무정보를 내부관리 목적으로 활용하고 있는 바 연결기업은 제약업 기타금융업 의료


## 5. 불용어 로딩과 추가

`data/raw/stopwords_ko_esg.txt` 외에 사업보고서에 흔한 일반 공시어와, 로마숫자가 위 4절에서 누락된 케이스를 위해 안전망으로 추가합니다.


In [5]:
with STOPWORDS_PATH.open('r', encoding='utf-8') as f:
    stopwords = {
        unicodedata.normalize('NFKC', line.strip()).lower()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

# 추가 일반 공시어 (Kiwi가 명사/용언으로 잡지만 ESG 신호가 아닌 단어)
GENERIC_CORPORATE = {
    '회사', '당사', '사업', '내용', '관련', '사항', '기준', '관리', '운영', '활동',
    '실시', '제도', '계획', '현황', '대상', '경우', '통해', '위해', '대한', '대하', '따르',
    '보고', '작성', '연결', '별도', '재무', '매출', '손익', '자산', '부채', '금액', '계약',
    '제품', '시장', '부문', '영업', '생산', '판매', '서비스', '종속', '관계', '변경', '증가', '감소',
    '찬성', '반대', '가결', '선임', '위원', '위원장', '총회', '후보',
}

# 로마숫자 안전망 (clean_text에서 빠졌더라도 Kiwi 토큰 단계에서 한 번 더 차단)
ROMAN_TOKENS = {
    'i', 'ii', 'iii', 'iv', 'v', 'vi', 'vii', 'viii', 'ix', 'x', 'xi', 'xii',
}

stopwords |= GENERIC_CORPORATE | ROMAN_TOKENS

print(f'불용어 총 개수: {len(stopwords)}')


불용어 총 개수: 84


## 6. Kiwi 형태소 분석 → `kiwi_text`

`raw_text`(=공백·문장 경계가 살아있는 원본)를 Kiwi에 넣고, 다음 토큰만 남깁니다.

- 품사: N* (명사), V* (용언), SL (영문)
- 길이 ≥ 2
- 한글 또는 영문만 (혼합/숫자 제외)
- 불용어 제외 (로마숫자 안전망 포함)

**원본의 ii/xi 버그 차단 지점**: clean_text 정규식(1차) + Kiwi 토큰 단계 불용어(2차)로 이중 방어합니다.


In [6]:
kiwi = Kiwi()
KEEP_POS_PREFIXES = ('N', 'V')
KEEP_POS_TAGS = {'SL'}
TOKEN_RE = re.compile(r'^[가-힣A-Za-z]+$')


def kiwi_tokens(text: str) -> list[str]:
    if not isinstance(text, str) or not text.strip():
        return []
    # raw에 있는 로마숫자도 미리 제거 (Kiwi가 SL로 인식하기 전에)
    text = unicodedata.normalize('NFKC', text)
    text = strip_section_markers(text)
    out = []
    for token in kiwi.tokenize(text):
        word = unicodedata.normalize('NFKC', token.form).lower().strip()
        if len(word) <= 1:
            continue
        if word in stopwords:
            continue
        if not TOKEN_RE.fullmatch(word):
            continue
        if not (token.tag.startswith(KEEP_POS_PREFIXES) or token.tag in KEEP_POS_TAGS):
            continue
        out.append(word)
    return out


df['kiwi_tokens'] = df['raw_text'].map(kiwi_tokens)
df['kiwi_text'] = df['kiwi_tokens'].map(' '.join)
df['token_count'] = df['kiwi_tokens'].map(len)

print(f'평균 토큰 수: {df["token_count"].mean():.0f}')
print(f'토큰 0개 행: {(df["token_count"] == 0).sum()}')

# 로마숫자 토큰이 남아 있는지 확인
all_tokens = pd.Series([t for tokens in df['kiwi_tokens'] for t in tokens])
leaked_roman = [t for t in all_tokens.unique() if t in ROMAN_TOKENS]
print(f'\n남은 로마숫자 토큰: {leaked_roman}')
print('\n샘플 kiwi_text 앞 30개 토큰:')
print(' '.join(df.loc[df['token_count'] > 0, 'kiwi_text'].iloc[0].split()[:30]))


평균 토큰 수: 14097
토큰 0개 행: 0

남은 로마숫자 토큰: []

샘플 kiwi_text 앞 30개 토큰:
개요 일반 지배 기업 실체 재화 용역 근거 구분 정보 내부 목적 활용 제약 의료 기기 제조 판매업 기타 금융업 영위 기타 금융업 중요 관점 비중 지배 개황 구분 실체


## 7. 최종 CSV 저장

요구사양에 맞춰 1개 CSV에 5개 열만 저장합니다.

| 열 | 의미 |
|---|---|
| `회사이름` | company_name |
| `esg_year` | KCGS 평가연도 (= fiscal_year + 1) |
| `raw_text` | 5개 섹션 결합 원문 |
| `clean_text` | 한글/영문/숫자 + 로마숫자 제거 + 공백 정규화 |
| `kiwi_text` | Kiwi 토큰화 + 불용어 제거 + 공백 구분 토큰 |


In [7]:
out = df[['회사이름', 'esg_year', 'raw_text', 'clean_text', 'kiwi_text']].copy()
out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'저장: {OUTPUT_CSV}')
print(f'행 수: {len(out)}')
print(f'열: {list(out.columns)}')

# 비어있는 행 확인
empty_rows = out['kiwi_text'].str.strip().eq('').sum()
print(f'kiwi_text 비어있는 행: {empty_rows} (수집 실패 등)')
out.head(3)


저장: /Users/home1/esg-dart-project/data/processed/dart_preprocessed.csv
행 수: 381
열: ['회사이름', 'esg_year', 'raw_text', 'clean_text', 'kiwi_text']
kiwi_text 비어있는 행: 0 (수집 실패 등)


,회사이름,esg_year,raw_text,clean_text,kiwi_text
0,동화약품,2023,II. 사업의 내용\n\n1. 사업의 개요\n\n1. 일반적인 사항지배기업인 연결실...,사업의 내용 1 사업의 개요 1 일반적인 사항지배기업인 연결실체는 제공하는 재화나 ...,개요 일반 지배 기업 실체 재화 용역 근거 구분 정보 내부 목적 활용 제약 의료 기...
1,동화약품,2024,II. 사업의 내용\n\n1. 사업의 개요\n\n1. 일반적인 사항지배기업인 연결실...,사업의 내용 1 사업의 개요 1 일반적인 사항지배기업인 연결실체는 제공하는 재화나 ...,개요 일반 지배 기업 실체 재화 용역 근거 구분 정보 내부 목적 활용 제약 의료 기...
2,동화약품,2025,II. 사업의 내용\n\n1. 사업의 개요\n\n1. 일반적인 사항지배기업인 연결실...,사업의 내용 1 사업의 개요 1 일반적인 사항지배기업인 연결실체는 제공하는 재화나 ...,개요 일반 지배 기업 실체 재화 용역 근거 구분 정보 내부 목적 활용 제약 의료 기...
